# HPO Stroke Prediction — GridSearchCV + MLflow

Ejecutar **después** del DAG `process_etl_stroke` en Airflow.

Referencia del modelo: [docs/MODELO.md](../docs/MODELO.md)

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import boto3
import awswrangler as wr
import mlflow

%env AWS_ACCESS_KEY_ID=minio
%env AWS_SECRET_ACCESS_KEY=minio123
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

In [ ]:
mlflow.set_tracking_uri('http://localhost:5001')

X_train = wr.s3.read_csv('s3://data/final/train/stroke_X_train.csv')
y_train = wr.s3.read_csv('s3://data/final/train/stroke_y_train.csv').squeeze()
X_test = wr.s3.read_csv('s3://data/final/test/stroke_X_test.csv')
y_test = wr.s3.read_csv('s3://data/final/test/stroke_y_test.csv').squeeze()

s3 = boto3.client('s3', endpoint_url='http://localhost:9000')
data_dict = json.loads(s3.get_object(Bucket='data', Key='data_info/data.json')['Body'].read())
feature_columns = data_dict['feature_columns']

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
from train import (
    run_grid_search,
    compute_youden_threshold,
    evaluate_model,
    register_champion_model,
)

search = run_grid_search(X_train, y_train, feature_columns)
best_model = search.best_estimator_
threshold = compute_youden_threshold(best_model, X_train, y_train)
metrics = evaluate_model(best_model, X_test, y_test, threshold)

print('Best params:', search.best_params_)
print('F1 CV:', round(search.best_score_, 3))
print('Youden threshold:', round(threshold, 3))
print('Test metrics:', {k: round(v, 3) for k, v in metrics.items()})

In [ ]:
# Actualizar umbral en MinIO
data_dict['optimal_threshold'] = threshold
s3.put_object(Bucket='data', Key='data_info/data.json', Body=json.dumps(data_dict, indent=2))

model_uri = register_champion_model(
    best_model,
    X_train,
    metrics,
    threshold,
    tracking_uri='http://localhost:5001',
)
print('Model registered:', model_uri)